In [12]:
# Data Visualization
#1.now we will do geospation analyis based on the neigbourhood data
#2.join with our citibike data,
#3.visualize top routes
#   A.change width
#   B.change color
#4.Coropleth Maps

#### Step 15: Geopandas
##### Read and Visualize jersey-city-neighborhoods.geojson data

In [13]:
import geopandas as gpd
from urllib.parse import urlencode
from pathlib import Path
import requests
import pandas as pd

import plotly.express as px
import folium

import os
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile


url = '../data/citibike/JC/jersey-city-neighborhoods.geojson'

jersey_city = gpd.read_file(url)

jersey_city.head()

,cartodb_id,area_sq_ft,acres,area,neighborhood,color,lon,lat,geometry
0,38,411601381.8,9449.068,Greenville,Port Liberte,NaN,-74.074540,40.694202,"POLYGON ((-74.06862 40.70098, -74.06808 40.696..."
1,52,411601381.8,9449.068,Bergen-Lafayette,LSP Industrial,NaN,-74.062358,40.699189,"POLYGON ((-74.06808 40.69684, -74.06862 40.700..."
2,29,411601381.8,9449.068,West Side,Hackensack,NaN,-74.085147,40.735520,"POLYGON ((-74.07601 40.73822, -74.07781 40.737..."
3,35,411601381.8,9449.068,Bergen-Lafayette,Lafayette,12.0,-74.061279,40.712676,"POLYGON ((-74.056 40.71735, -74.056 40.71692, ..."
4,51,411601381.8,9449.068,Greenville,Jackson Hill,15.0,-74.085503,40.700791,"POLYGON ((-74.07561 40.70233, -74.0758 40.7020..."


In [14]:
jersey_city.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   cartodb_id    53 non-null     int32   
 1   area_sq_ft    53 non-null     float64 
 2   acres         53 non-null     float64 
 3   area          53 non-null     str     
 4   neighborhood  53 non-null     str     
 5   color         18 non-null     float64 
 6   lon           53 non-null     float64 
 7   lat           53 non-null     float64 
 8   geometry      53 non-null     geometry
dtypes: float64(5), geometry(1), int32(1), str(2)
memory usage: 3.6 KB


In [15]:
jersey_city.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [16]:
citibike_df = pd.read_csv('../data/citibike/JC/JC_Enriched.csv')

In [17]:
#station_departures
start_stations = citibike_df[
    [
        "ride_id",
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lng"
    ]
].copy()

In [18]:
start_stations = start_stations.rename(
    columns={
        "start_station_id": "station_id",
        "start_station_name": "station_name",
        "start_lat": "lat",
        "start_lng": "lng"
    }
)

start_stations["activity_type"] = "departure"

In [19]:
#station_arrivals
end_stations = citibike_df[
    [
        "ride_id",
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lng"
    ]
].copy()

end_stations = end_stations.rename(
    columns={
        "end_station_id": "station_id",
        "end_station_name": "station_name",
        "end_lat": "lat",
        "end_lng": "lng"
    }
)

end_stations["activity_type"] = "arrival"

end_stations.head()

,ride_id,station_id,station_name,lat,lng,activity_type
0,40FBC7AEEBB220EC,6233.04,Pier 61 at Chelsea Piers,40.746872,-74.00821,arrival
1,47FB1CA65D3F7E0F,HB402,Madison St & 1 St,40.738790,-74.03930,arrival
2,BFB10DE1E40F3709,HB402,Madison St & 1 St,40.738790,-74.03930,arrival
3,F16B896495AAF618,HB402,Madison St & 1 St,40.738790,-74.03930,arrival
4,76D77D5F1D491B3C,HB402,Madison St & 1 St,40.738790,-74.03930,arrival


#### Concatenate Departures and Arrivals


In [20]:
station_activity_long = pd.concat(
    [
        start_stations,
        end_stations
    ],
    ignore_index=True
)

station_activity_long = station_activity_long.dropna(
    subset=[
        "station_id",
        "station_name",
        "lat",
        "lng"
    ]
)

station_activity_long.head()

,ride_id,station_id,station_name,lat,lng,activity_type
0,40FBC7AEEBB220EC,HB202,14 St Ferry - 14 St & Shipyard Ln,40.752961,-74.024353,departure
1,47FB1CA65D3F7E0F,HB202,14 St Ferry - 14 St & Shipyard Ln,40.752961,-74.024353,departure
2,BFB10DE1E40F3709,HB601,Church Sq Park - 5 St & Park Ave,40.742659,-74.032233,departure
3,F16B896495AAF618,HB402,Madison St & 1 St,40.738790,-74.039300,departure
4,76D77D5F1D491B3C,HB303,Clinton St & 7 St,40.745420,-74.033320,departure


#### Cleaning and Aggregating


In [21]:
station_activity_agg = (
    station_activity_long
    .groupby(
        [
            "station_id",
            "station_name",
            "lat",
            "lng",
            "activity_type"
        ],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

station_activity_agg.head()

,station_id,station_name,lat,lng,activity_type,number_of_rides
0,3300.04,E 49 St & Church Ave,40.651820,-73.931350,arrival,2
1,4298.05,Degraw St & Smith St,40.682915,-73.993182,arrival,2
2,4371.01,Warren St & Smith St,40.685424,-73.991278,arrival,2
3,4413.08,Warren St & Court St,40.686371,-73.993833,arrival,4
4,4437.01,3 Ave & Schermerhorn St,40.686832,-73.979677,arrival,2


##### Pivot the Data


In [22]:
#Pivot to One Row per Station
#Now we convert departure and arrival into separate columns.
station_summary = (
    station_activity_agg
    .pivot_table(
        index=[
            "station_id",
            "station_name",
            "lat",
            "lng"
        ],
        columns="activity_type",
        values="number_of_rides",
        fill_value=0
    )
    .reset_index()
)

station_summary.head()

activity_type,station_id,station_name,lat,lng,arrival,departure
0,3300.04,E 49 St & Church Ave,40.651820,-73.931350,2.0,0.0
1,4298.05,Degraw St & Smith St,40.682915,-73.993182,2.0,0.0
2,4371.01,Warren St & Smith St,40.685424,-73.991278,2.0,0.0
3,4413.08,Warren St & Court St,40.686371,-73.993833,4.0,0.0
4,4437.01,3 Ave & Schermerhorn St,40.686832,-73.979677,2.0,0.0


In [23]:
#Rename and Create Final Metrics
station_summary = station_summary.rename(
    columns={
        "departure": "total_departures",
        "arrival": "total_arrivals"
    }
)

station_summary["total_activity"] = (
    station_summary["total_departures"] +
    station_summary["total_arrivals"]
)

station_summary["net_departures"] = (
    station_summary["total_departures"] -
    station_summary["total_arrivals"]
)

station_summary = station_summary.sort_values(
    "total_activity",
    ascending=False
).reset_index(drop=True)

station_summary.head()

activity_type,station_id,station_name,lat,lng,total_arrivals,total_departures,total_activity,net_departures
0,JC115,Grove St PATH,40.719410,-74.043090,38846.0,38162.0,77008.0,-684.0
1,HB106,River St & Newark St,40.736722,-74.029007,30436.0,29018.0,59454.0,-1418.0
2,HB101,Hoboken Terminal - Hudson St & Hudson Pl,40.735938,-74.030305,22050.0,21826.0,43876.0,-224.0
3,JC109,Bergen Ave & Sip Ave,40.731009,-74.064437,19450.0,19696.0,39146.0,246.0
4,JC009,Hamilton Park,40.727596,-74.044247,17976.0,17882.0,35858.0,-94.0


In [24]:
station_gdf = gpd.GeoDataFrame(
    station_summary,
    geometry=gpd.points_from_xy(
        station_summary["lng"],
        station_summary["lat"]
    ),
    crs="EPSG:4326"
)

station_gdf.head()

activity_type,station_id,station_name,lat,lng,total_arrivals,total_departures,total_activity,net_departures,geometry
0,JC115,Grove St PATH,40.719410,-74.043090,38846.0,38162.0,77008.0,-684.0,POINT (-74.04309 40.71941)
1,HB106,River St & Newark St,40.736722,-74.029007,30436.0,29018.0,59454.0,-1418.0,POINT (-74.02901 40.73672)
2,HB101,Hoboken Terminal - Hudson St & Hudson Pl,40.735938,-74.030305,22050.0,21826.0,43876.0,-224.0,POINT (-74.0303 40.73594)
3,JC109,Bergen Ave & Sip Ave,40.731009,-74.064437,19450.0,19696.0,39146.0,246.0,POINT (-74.06444 40.73101)
4,JC009,Hamilton Park,40.727596,-74.044247,17976.0,17882.0,35858.0,-94.0,POINT (-74.04425 40.7276)


In [25]:
#Create Route-Level Summary
route_summary = (
    citibike_df.dropna(
        subset=[
            "start_station_id",
            "start_station_name",
            "start_station_id",
            "start_station_name",
            "start_lat",
            "start_lng",
            "end_lat",
            "end_lng"
        ]
    )
    .groupby(
        [
            "start_station_id",
            "start_station_name",
            "end_station_id",
            "end_station_name",
            "start_lat",
            "start_lng",
            "end_lat",
            "end_lng"

        ],
        as_index=False
    )
    .agg(
        number_of_rides=("ride_id", "count")
    )
    .sort_values("number_of_rides", ascending=False)
)

route_summary["route"] = (
    route_summary["start_station_name"] +
    " → " +
    route_summary["end_station_name"]
)

route_summary.head()

,start_station_id,start_station_name,end_station_id,end_station_name,start_lat,start_lng,end_lat,end_lng,number_of_rides,route
5250,JC055,McGinley Square,JC109,Bergen Ave & Sip Ave,40.725340,-74.067622,40.731009,-74.064437,4178,McGinley Square → Bergen Ave & Sip Ave
79,HB101,Hoboken Terminal - Hudson St & Hudson Pl,JC105,Hoboken Ave at Monmouth St,40.735938,-74.030305,40.735208,-74.046964,3362,Hoboken Terminal - Hudson St & Hudson Pl → Hob...
7861,JC109,Bergen Ave & Sip Ave,JC055,McGinley Square,40.731009,-74.064437,40.725340,-74.067622,3326,Bergen Ave & Sip Ave → McGinley Square
8085,JC115,Grove St PATH,JC013,Marin Light Rail,40.719410,-74.043090,40.714584,-74.042817,3188,Grove St PATH → Marin Light Rail
3653,JC013,Marin Light Rail,JC115,Grove St PATH,40.714584,-74.042817,40.719410,-74.043090,2676,Marin Light Rail → Grove St PATH


In [26]:
#Inspect Top Routes
top_routes = route_summary.head(20)

top_routes[
    [
        "route",
        "number_of_rides"
    ]
]

,route,number_of_rides
5250,McGinley Square → Bergen Ave & Sip Ave,4178
79,Hoboken Terminal - Hudson St & Hudson Pl → Hob...,3362
7861,Bergen Ave & Sip Ave → McGinley Square,3326
8085,Grove St PATH → Marin Light Rail,3188
3653,Marin Light Rail → Grove St PATH,2676
7609,Hoboken Ave at Monmouth St → Hoboken Terminal ...,2620
4285,Brunswick St → Grove St PATH,2568
8134,Grove St PATH → Pacific Ave & Communipaw Ave,2492
14,Hoboken Terminal - Hudson St & Hudson Pl → Mad...,2426
13,Hoboken Terminal - Hudson St & Hudson Pl → Sou...,2420


In [27]:
#Drawing Top Lines with Folium
import folium

top_lines = route_summary.head(100).copy()

center_lat = station_gdf["lat"].mean()
center_lng = station_gdf["lng"].mean()

line_map = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12
)

max_rides = top_lines["number_of_rides"].max()

for _, row in top_lines.iterrows():

    start_point = [
        row["start_lat"],
        row["start_lng"]
    ]

    end_point = [
        row["end_lat"],
        row["end_lng"]
    ]

    line_weight = 1 + (row["number_of_rides"] / max_rides) * 8

    folium.PolyLine(
        locations=[start_point, end_point],
        weight=line_weight,
        opacity=0.5,
        popup=f"""
        <b>{row['route']}</b><br>
        Number of Rides: {row['number_of_rides']}
        """
    ).add_to(line_map)

line_map

In [28]:
#Check CRS Before Spatial Join
print("Station CRS:", station_gdf.crs)
print("Neighborhood CRS:", jersey_city.crs)

Station CRS: EPSG:4326
Neighborhood CRS: EPSG:4326


In [29]:
station_gdf = station_gdf.to_crs("EPSG:4326")
jersey_city = jersey_city.to_crs("EPSG:4326")

In [30]:
#Inspect Neighborhood Columns
jersey_city.columns

Index(['cartodb_id', 'area_sq_ft', 'acres', 'area', 'neighborhood', 'color',
       'lon', 'lat', 'geometry'],
      dtype='str')

In [31]:
#Spatial Join: Assign Stations to Neighborhoods
station_neighborhood = gpd.sjoin(
    station_gdf,
    jersey_city,
    how="inner",
    predicate="within"
)

station_neighborhood.head()

,station_id,station_name,lat_left,lng,total_arrivals,total_departures,total_activity,net_departures,geometry,index_right,cartodb_id,area_sq_ft,acres,area,neighborhood,color,lon,lat_right
0,JC115,Grove St PATH,40.719410,-74.043090,38846.0,38162.0,77008.0,-684.0,POINT (-74.04309 40.71941),49,2,411601381.8,9449.068,Downtown,Van Vorst Park,21.0,-74.047234,40.718943
3,JC109,Bergen Ave & Sip Ave,40.731009,-74.064437,19450.0,19696.0,39146.0,246.0,POINT (-74.06444 40.73101),16,31,411601381.8,9449.068,Journal Square,Journal Square,NaN,-74.063466,40.733757
4,JC009,Hamilton Park,40.727596,-74.044247,17976.0,17882.0,35858.0,-94.0,POINT (-74.04425 40.7276),10,18,411601381.8,9449.068,Downtown,Hamilton Park,28.0,-74.046672,40.727436
6,JC116,Exchange Pl,40.716366,-74.034344,16104.0,16250.0,32354.0,146.0,POINT (-74.03434 40.71637),48,13,411601381.8,9449.068,Downtown,Exchange Place,23.0,-74.033539,40.716458
12,JC066,Newport PATH,40.727224,-74.033759,14168.0,14216.0,28384.0,48.0,POINT (-74.03376 40.72722),8,12,411601381.8,9449.068,Downtown,Newport,22.0,-74.034927,40.729255


In [32]:
#Check How Many Stations Are Inside Jersey City
print("All stations:", len(station_gdf))
print("Stations inside Jersey City neighborhoods:", len(station_neighborhood))

All stations: 373
Stations inside Jersey City neighborhoods: 78


In [33]:
#Neighborhood-Level Summary
neighborhood_activity = (
    station_neighborhood
    .groupby('neighborhood', as_index=False)
    .agg(
        number_of_stations=("station_id", "nunique"),
        total_departures=("total_departures", "sum"),
        total_arrivals=("total_arrivals", "sum"),
        total_activity=("total_activity", "sum"),
        net_departures=("net_departures", "sum")
    )
)

neighborhood_activity["avg_activity_per_station"] = (
    neighborhood_activity["total_activity"] /
    neighborhood_activity["number_of_stations"]
)

neighborhood_activity = neighborhood_activity.sort_values(
    "total_activity",
    ascending=False
)

neighborhood_activity.head()

,neighborhood,number_of_stations,total_departures,total_arrivals,total_activity,net_departures,avg_activity_per_station
25,Van Vorst Park,6,81416.0,82782.0,164198.0,-1366.0,27366.333333
17,Palus Hook,6,44758.0,44210.0,88968.0,548.0,14828.000000
11,Lafayette,6,32382.0,32512.0,64894.0,-130.0,10815.666667
10,Journal Square,3,28390.0,27874.0,56264.0,516.0,18754.666667
14,Newport,2,27418.0,27318.0,54736.0,100.0,27368.000000


In [34]:
#Create Center Point for the Map
center_lat = station_gdf.geometry.y.mean()
center_lng = station_gdf.geometry.x.mean()

center_lat, center_lng

(np.float64(40.73458211583215), np.float64(-74.00873125371946))

In [35]:
#Visualize Each Station as a Point
station_point_map = folium.Map(
    location=[center_lat, center_lng],
    zoom_start=12
)

for _, row in station_gdf.iterrows():

    folium.CircleMarker(
        location=[
            row.geometry.y,
            row.geometry.x
        ],
        radius=5,
        popup=f"""
        <b>{row['station_name']}</b><br>
        Station ID: {row['station_id']}<br>
        Departures: {row['total_departures']:.0f}<br>
        Arrivals: {row['total_arrivals']:.0f}<br>
        Total Activity: {row['total_activity']:.0f}<br>
        Net Departures: {row['net_departures']:.0f}
        """,
        tooltip=row["station_name"],
        fill=True,
        fill_opacity=0.6,
        opacity=0.8
    ).add_to(station_point_map)

station_point_map

In [36]:
#Merge Neighborhood Metrics Back to Polygons
neighborhood_choropleth_gdf = jersey_city.merge(
    neighborhood_activity,
    on='neighborhood',
    how="left"
)

neighborhood_choropleth_gdf.head()

,cartodb_id,area_sq_ft,acres,area,neighborhood,color,lon,lat,geometry,number_of_stations,total_departures,total_arrivals,total_activity,net_departures,avg_activity_per_station
0,38,411601381.8,9449.068,Greenville,Port Liberte,NaN,-74.074540,40.694202,"POLYGON ((-74.06862 40.70098, -74.06808 40.696...",1.0,828.0,842.0,1670.0,-14.0,1670.000000
1,52,411601381.8,9449.068,Bergen-Lafayette,LSP Industrial,NaN,-74.062358,40.699189,"POLYGON ((-74.06808 40.69684, -74.06862 40.700...",NaN,NaN,NaN,NaN,NaN,NaN
2,29,411601381.8,9449.068,West Side,Hackensack,NaN,-74.085147,40.735520,"POLYGON ((-74.07601 40.73822, -74.07781 40.737...",NaN,NaN,NaN,NaN,NaN,NaN
3,35,411601381.8,9449.068,Bergen-Lafayette,Lafayette,12.0,-74.061279,40.712676,"POLYGON ((-74.056 40.71735, -74.056 40.71692, ...",6.0,32382.0,32512.0,64894.0,-130.0,10815.666667
4,51,411601381.8,9449.068,Greenville,Jackson Hill,15.0,-74.085503,40.700791,"POLYGON ((-74.07561 40.70233, -74.0758 40.7020...",15.0,10626.0,10748.0,21374.0,-122.0,1424.933333


In [37]:
activity_columns = [
    "number_of_stations",
    "total_departures",
    "total_arrivals",
    "total_activity",
    "net_departures",
    "avg_activity_per_station"
]

neighborhood_choropleth_gdf[activity_columns] = (
    neighborhood_choropleth_gdf[activity_columns]
    .fillna(0)
)

neighborhood_choropleth_gdf[
    [
        'neighborhood',
        "number_of_stations",
        "total_departures",
        "total_arrivals",
        "total_activity",
        "avg_activity_per_station"
    ]
].head()

,neighborhood,number_of_stations,total_departures,total_arrivals,total_activity,avg_activity_per_station
0,Port Liberte,1.0,828.0,842.0,1670.0,1670.000000
1,LSP Industrial,0.0,0.0,0.0,0.0,0.000000
2,Hackensack,0.0,0.0,0.0,0.0,0.000000
3,Lafayette,6.0,32382.0,32512.0,64894.0,10815.666667
4,Jackson Hill,15.0,10626.0,10748.0,21374.0,1424.933333


In [38]:
#Choropleth Map with Folium
def create_neighborhood_choropleth(
    gdf,
    metric,
    legend_name,
    neighborhood_col="neighborhood"
):
    choropleth_map = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=12
    )

    folium.Choropleth(
        geo_data=gdf,
        data=gdf,
        columns=[neighborhood_col, metric],
        key_on=f"feature.properties.{neighborhood_col}",
        fill_opacity=0.7,
        line_opacity=0.4,
        legend_name=legend_name,
        nan_fill_opacity=0.1
    ).add_to(choropleth_map)

    folium.GeoJson(
        gdf,
        name="Neighborhood Boundaries",
        tooltip=folium.GeoJsonTooltip(
            fields=[
                neighborhood_col,
                metric
            ],
            aliases=[
                "Neighborhood:",
                f"{legend_name}:"
            ],
            localize=True
        ),
        style_function=lambda feature: {
            "fillOpacity": 0,
            "color": "black",
            "weight": 1
        }
    ).add_to(choropleth_map)

    folium.LayerControl().add_to(choropleth_map)

    return choropleth_map

In [39]:
#Helper Function for Folium Choropleth
def create_neighborhood_choropleth(
    gdf,
    metric,
    legend_name,
    neighborhood_col="neighborhood"
):
    choropleth_map = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=12
    )

    folium.Choropleth(
        geo_data=gdf,
        data=gdf,
        columns=[neighborhood_col, metric],
        key_on=f"feature.properties.{neighborhood_col}",
        fill_opacity=0.7,
        line_opacity=0.4,
        legend_name=legend_name,
        nan_fill_opacity=0.1
    ).add_to(choropleth_map)

    folium.GeoJson(
        gdf,
        name="Neighborhood Boundaries",
        tooltip=folium.GeoJsonTooltip(
            fields=[
                neighborhood_col,
                metric
            ],
            aliases=[
                "Neighborhood:",
                f"{legend_name}:"
            ],
            localize=True
        ),
        style_function=lambda feature: {
            "fillOpacity": 0,
            "color": "black",
            "weight": 1
        }
    ).add_to(choropleth_map)

    folium.LayerControl().add_to(choropleth_map)

    return choropleth_map

In [40]:
#Choropleth: Total Activity by Neighborhood
total_activity_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="total_activity",
    legend_name="Total Citi Bike Activity",
    neighborhood_col="neighborhood"
)

total_activity_map

In [41]:
#Choropleth by Number of Stations
station_count_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="number_of_stations",
    legend_name="Number of Citi Bike Stations",
    neighborhood_col="neighborhood"
)

station_count_map

In [42]:
#Choropleth by Number of Stations
station_count_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="number_of_stations",
    legend_name="Number of Citi Bike Stations",
    neighborhood_col="neighborhood"
)

station_count_map

In [43]:
#Choropleth by Average Activity per Station
avg_activity_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="avg_activity_per_station",
    legend_name="Average Activity per Station",
    neighborhood_col="neighborhood"
)

avg_activity_map

In [44]:
#Choropleth by Total Departures
departures_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="total_departures",
    legend_name="Total Departures",
    neighborhood_col="neighborhood"
)

departures_map

In [45]:
#Choropleth by Total Arrivals
arrivals_map = create_neighborhood_choropleth(
    gdf=neighborhood_choropleth_gdf,
    metric="total_arrivals",
    legend_name="Total Arrivals",
    neighborhood_col="neighborhood"
)

arrivals_map